In [1]:
# Import all functions from the required modules
from cordo_sherpa_module import *
from cordo_chimere_module import *
from expo_functions_module import *
from mortality_analysis_module import *
from association_module import *
print("Successfully loaded all modules")

loaded defined RR values
Successfully loaded mean conc command
Successfully loaded all modules


In [2]:
# Paths to the files
path_fichier_shp = "data/2-output-data/donnees_shp"
title_shp = "donnees_insee_iris"
path_fichier_pourcents = "data/2-output-data"
title_pourcents = "pourcents"

# Load the concentration points
conc_points = coordo_chimere(pol="ug_NO2", year=2019, SC="s1".upper())
# Load the exported data
donnees_exportees = gpd.read_file(os.path.join(path_fichier_shp, f"{title_shp}.shp"))

# Transform the CRS of the exported data to match the concentration points
donnees_exportees_transformed = donnees_exportees.to_crs(epsg=conc_points.crs.to_epsg())

# Check if CRSs are the same
if conc_points.crs == donnees_exportees_transformed.crs:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are the same.")
else:
    print("CRS for conc_points_transformed and donnees_exportees_transformed are different.")

Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
CRS for conc_points_transformed and donnees_exportees_transformed are the same.


In [3]:
import os
# Define paths for shapefiles
path_fichier_shp = "data/2-output-data/donnees_shp"
path_fichier_shp_1 = "data/2-output-data/donnees_shp_1"
path_fichier_shp_2 = "data/2-output-data/donnees_shp_2"
path_fichier_shp_3 = "data/2-output-data/donnees_shp_3"
path_fichier_pourcents = "data/2-output-data"

# Titles for INSEE Data
title_shp = "donnees_insee_iris"
title_shp_1 = "donnees_insee_iris_toutage_1"
title_shp_2 = "donnees_insee_iris_toutage_2"
title_shp_3 = "donnees_insee_iris_toutage_3"
title_pourcents = "pourcents"

# Read shapefiles into GeoDataFrames
donnees_shp_1 = gpd.read_file(os.path.join(path_fichier_shp_1, f"{title_shp_1}.shp"))
donnees_shp_2 = gpd.read_file(os.path.join(path_fichier_shp_2, f"{title_shp_2}.shp"))
donnees_shp_3 = gpd.read_file(os.path.join(path_fichier_shp_3, f"{title_shp_3}.shp"))

# Combine the three GeoDataFrames
donnees_merged = gpd.GeoDataFrame(pd.concat([donnees_shp_1, donnees_shp_2, donnees_shp_3], ignore_index=True))
print(donnees_merged.head())

grille_combinee = gpd.read_file(os.path.join(path_fichier_pourcents, f"{title_pourcents}.shp"))
grille_combinee = grille_combinee.to_crs(conc_points.crs)

     iriscod irisname comcod comname depcod depname  regcod           regname  \
0  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
1  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
2  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
3  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   
4  721910000    Mayet  72191   Mayet     72  Sarthe      52  Pays de la Loire   

   age    pop2019    pop2030    pop2050  mort2019  mort2030  mort2050  \
0    0  31.979362  30.560224  28.974134  0.114652  0.101665  0.085457   
1    1  32.735555  30.384526  29.200275  0.018369  0.016510  0.014105   
2    2  33.511730  30.265005  29.417218  0.008333  0.007299  0.006310   
3    3  34.431724  30.206960  29.615683  0.005018  0.004605  0.004083   
4    4  35.587474  30.214303  29.787011  0.003853  0.003519  0.003135   

                                            geometry  
0  POLYGON ((497887

In [7]:
#Helper functions to plot maps for PM2.5 and NO2 concentrations
def pretty_pol(pol):
    return "PM2.5" if pol == "ug_PM25_RH50" else "NO2"

def _square_extent_from_bounds(bounds, pad_frac=0.02):
    xmin, ymin, xmax, ymax = bounds
    dx = max(xmax - xmin, 1e-12)
    dy = max(ymax - ymin, 1e-12)
    cx = (xmin + xmax) / 2.0
    cy = (ymin + ymax) / 2.0
    half = max(dx, dy) / 2.0
    half = half * (1.0 + pad_frac)
    return (cx - half, cx + half, cy - half, cy + half)

def _get_iris_code_col(gdf):
    for c in ["CODE_IRIS", "iriscod", "code_iris", "IRIS_CODE", "IRIS", "iris"]:
        if c in gdf.columns:
            return c
    return None

def _points_to_iris_polygons(points_gdf, value_col, iris_polys, iris_code_col, fill_missing=True):
    pts = points_gdf[[value_col, "geometry"]].copy()
    if pts.crs != iris_polys.crs:
        pts = pts.to_crs(iris_polys.crs)

    joined = gpd.sjoin(pts, iris_polys[[iris_code_col, "geometry"]], how="left", predicate="within")

    missing_pts = joined[iris_code_col].isna()
    if bool(missing_pts.any()):
        try:
            nearest = gpd.sjoin_nearest(
                pts.loc[missing_pts],
                iris_polys[[iris_code_col, "geometry"]],
                how="left",
                distance_col="__dist",
            )
            joined.loc[missing_pts, iris_code_col] = nearest[iris_code_col].values
        except Exception:
            pass

    agg = (
        joined.dropna(subset=[iris_code_col])
        .groupby(iris_code_col)[value_col]
        .mean()
        .reset_index()
    )

    out = iris_polys.merge(agg, on=iris_code_col, how="left")

    if fill_missing and out[value_col].isna().any():
        s = pd.to_numeric(out[value_col], errors="coerce").astype(float)
        if s.notna().any():
            out[value_col] = s.fillna(s.mean()).values

    return out

def get_pw_mean_commune(iris_vals_gdf, val_col, pop_commune):
    """
    Population-weighted mean by commune:
    - mean(val_col) across IRIS within each comcod
    - weight commune means by commune population
    """
    commune_vals = iris_vals_gdf.groupby("comcod")[val_col].mean().reset_index()
    merged = pd.merge(commune_vals, pop_commune, on="comcod", how="inner")
    total_pop = merged["pop_val"].sum()
    if total_pop == 0:
        return 0.0
    return float((merged[val_col] * merged["pop_val"]).sum() / total_pop)

# -------------------------
# map plot function (polygons)
# -------------------------
def fast_plot(ax, poly_gdf, col, title, cmap, norm, extent, pw_mean=None, unit_label="µg/m³"):
    poly_gdf.plot(
        column=col,
        ax=ax,
        cmap=cmap,
        norm=norm,
        linewidth=0.0,
        edgecolor="none",
        missing_kwds={"color": "lightgrey", "edgecolor": "none"},
        rasterized=True,
    )

    title_text = f"{title}"
    if pw_mean is not None:
        title_text += f"\n$\\mathbf{{Population\\ Weighted\\ Mean\\ =\\ {pw_mean:.2f}\\ {unit_label}}}$"
    ax.set_title(title_text, pad=10)
    xmin, xmax, ymin, ymax = extent
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    ax.set_aspect("equal", adjustable="box")

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.tick_params(axis="both", which="major", length=4, width=0.8, direction="out")
    for spine in ax.spines.values():
        spine.set_linewidth(0.8)

    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    return sm

In [9]:
#-----------------------------------------------------------
# Codes to plot CHIMERE maps for PM2.5 and NO2 concentrations
# Task: Assign CHIMERE point values to IRIS polygons (from CONTOURS-IRIS shapefile),
#       aggregate to polygon level, then plot polygons (continuous choropleth)
#---------------------------------------------------------
import os
import logging
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm

# Define scenarios, pollutants, and years
scenarios = ["s2", "s3"]
pollutants = ["ug_PM25_RH50", "ug_NO2"]
years = ["2030", "2050"]

OUTPUT_DIR = "data/2-output-data/maps_CHIMERE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Global settings for map plots
plt.rcParams.update({
    "figure.dpi": 300,
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12
})

# Load IRIS shapefile (polygons used for plotting)
shp_path = "data/1-processed-data/CONTOURS-IRIS/CONTOURS-IRIS.shp"
iris = gpd.read_file(shp_path)

# -------------------------
# Main plotting function
# -------------------------
def plot_maps_for_pollutant_year(pol, year):
    pol_lbl = pretty_pol(pol)
    logging.info(f"Plotting {pol_lbl} - {year}")

    data = {}

    iris_code_col = _get_iris_code_col(iris)
    if iris_code_col is None:
        raise ValueError("IRIS shapefile does not contain a recognizable IRIS code column (e.g., CODE_IRIS).")

    # ---- Prepare population data aggregated at commune level (comcod)
    pop_data = donnees_exportees_transformed.copy()
    pop_col = f"pop{year}"
    pop_data[pop_col] = pd.to_numeric(pop_data[pop_col], errors="coerce").fillna(0)

    # Aggregate population at commune level
    pop_commune = pop_data.groupby("comcod")[pop_col].sum().reset_index()
    pop_commune.rename(columns={pop_col: "pop_val"}, inplace=True)

    def get_pw_mean_from_poly(target_poly_gdf, val_col):
        tmp = target_poly_gdf[[val_col, "comcod"]].copy()
        tmp[val_col] = pd.to_numeric(tmp[val_col], errors="coerce").astype(float)
        commune_vals = tmp.groupby("comcod")[val_col].mean().reset_index()
        merged = pd.merge(commune_vals, pop_commune, on="comcod", how="inner")
        total_pop = merged["pop_val"].sum()
        if total_pop == 0:
            return 0.0
        return float((merged[val_col] * merged["pop_val"]).sum() / total_pop)

    # ---- Load data once per scenario (assign to polygons before plotting)
    all_conc_vals = []
    all_delta_vals = []
    extent = None

    for sc in scenarios:
        logging.info(f"  Loading {sc.upper()}")
        scen_pts = coordo_chimere(pol, year=year, SC=sc.upper())
        base = coordo_ineris_chimere(pol, year="2019")
        corr_pts = correction_chimere(scen_pts, base)

        iris_polys = iris.copy()
        if scen_pts.crs is not None and iris_polys.crs != scen_pts.crs:
            iris_polys = iris_polys.to_crs(scen_pts.crs)

        # Attach comcod onto the IRIS polygons using donnees_exportees_transformed mapping
        iris_to_com = (
            donnees_exportees_transformed[["iriscod", "comcod"]]
            .drop_duplicates("iriscod")
            .rename(columns={"iriscod": iris_code_col})
        )
        iris_polys = iris_polys.merge(iris_to_com, on=iris_code_col, how="left")

        scen_poly = _points_to_iris_polygons(scen_pts, "conc", iris_polys, iris_code_col, fill_missing=True)
        corr_poly = _points_to_iris_polygons(corr_pts, "delta_conc", iris_polys, iris_code_col, fill_missing=True)

        pw_mean_conc = get_pw_mean_from_poly(scen_poly, "conc")
        pw_mean_delta = get_pw_mean_from_poly(corr_poly, "delta_conc")

        data[sc] = {
            "scen": scen_poly,
            "corr": corr_poly,
            "pw_mean_conc": pw_mean_conc,
            "pw_mean_delta": pw_mean_delta
        }

        all_conc_vals.append(pd.to_numeric(scen_poly["conc"], errors="coerce").astype(float).values)
        all_delta_vals.append(pd.to_numeric(corr_poly["delta_conc"], errors="coerce").astype(float).values)

        if extent is None:
            extent = _square_extent_from_bounds(iris_polys.total_bounds, pad_frac=0.02)

    # ---- Shared color scales
    all_conc = np.concatenate(all_conc_vals)
    all_delta = np.concatenate(all_delta_vals)

    conc_norm = Normalize(vmin=np.nanmin(all_conc), vmax=np.nanmax(all_conc))
    dmax = np.nanmax(np.abs(all_delta))
    delta_norm = TwoSlopeNorm(vcenter=0, vmin=-dmax, vmax=dmax)

    # ---- figure layout
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    fig.subplots_adjust(left=0.09, right=0.95, top=0.88, bottom=0.06, wspace=0.00, hspace=0.50)
    fig.suptitle(f"{pol_lbl} — {year} (S2 and S3)", fontsize=18)

    for i, sc in enumerate(scenarios):
        scen = data[sc]["scen"]
        corr = data[sc]["corr"]

        # Concentration map (polygons)
        sc_plot = fast_plot(
            axes[i, 0],
            scen,
            "conc",
            f"{sc.upper()} — Average Concentration",
            "viridis",
            conc_norm,
            extent,
            pw_mean=data[sc]["pw_mean_conc"]
        )

        # Delta map (polygons)
        delta_plot = fast_plot(
            axes[i, 1],
            corr,
            "delta_conc",
            f"{sc.upper()} — Δ Concentration",
            "coolwarm",
            delta_norm,
            extent,
            pw_mean=data[sc]["pw_mean_delta"]
        )

        # ---- Proper publication-style colorbars
        cbar_ax1 = axes[i, 0].inset_axes([1.02, 0.08, 0.035, 0.84])
        cb1 = plt.colorbar(sc_plot, cax=cbar_ax1)
        cb1.set_label("Mean Conc (µg/m³)", fontsize=12)
        cb1.ax.tick_params(labelsize=14)

        cbar_ax2 = axes[i, 1].inset_axes([1.02, 0.08, 0.035, 0.84])
        cb2 = plt.colorbar(delta_plot, cax=cbar_ax2)
        cb2.set_label("Δ Conc (µg/m³)", fontsize=14)
        cb2.ax.tick_params(labelsize=12)

    # ---- Save
    out_path = os.path.join(OUTPUT_DIR, f"pop_weighted_{pol_lbl}_{year}_S2_and_S3.png")
    fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)

    logging.info(f"Saved: {out_path}")


# -------------------------
# Main driver
# -------------------------
if __name__ == "__main__":
    for pol in pollutants:
        for year in years:
            plot_maps_for_pollutant_year(pol, year)

    logging.info("✅ All maps generated with population-weighted formatting.")


2026-02-10 11:54:36,594 - INFO - INFO - Plotting PM2.5 - 2030
2026-02-10 11:54:36,614 - INFO - INFO -   Loading S2


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS2_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_ineris function


C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
2026-02-10 11:54:40,328 - INFO - INFO -   Loading S3


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS3_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_ineris function


C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
2026-02-10 11:55:14,291 - INFO - INFO - Saved: data/2-output-data/maps_CHIMERE\pop_weighted_PM2.5_2030_S2_and_S3.png
2026-02-10 11:55:14,292 - INFO - INFO - Plotting PM2.5 - 2050
2026-02-10 11:55:14,313 - INFO - INFO -   Loading S2


...

C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
2026-02-10 11:56:27,787 - INFO - INFO - Saved: data/2-output-data/maps_CHIMERE\pop_weighted_NO2_2030_S2_and_S3.png
2026-02-10 11:56:27,788 - INFO - INFO - Plotting NO2 - 2050
2026-02-10 11:56:27,810 - INFO - INFO -   Loading S2


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2050_scenS2_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_ineris function


C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
2026-02-10 11:56:34,791 - INFO - INFO -   Loading S3


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2050_scenS3_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function
Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_ineris function


C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
2026-02-10 11:57:08,533 - INFO - INFO - Saved: data/2-output-data/maps_CHIMERE\pop_weighted_NO2_2050_S2_and_S3.png
2026-02-10 11:57:08,534 - INFO - INFO - ✅ All maps generated with population-weighted formatting.


In [8]:
#-----------------------------------------------------------
# Codes to plot CHIMERE maps for PM2.5 and NO2 concentrations
# Task: Assign CHIMERE point values to IRIS polygons (from CONTOURS-IRIS shapefile),
#       aggregate to polygon level, then plot polygons (continuous choropleth)
#---------------------------------------------------------
import os
import logging
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm

# Define scenarios, pollutants, and years
scenarios = ["s2", "s3"]
pollutants = ["ug_PM25_RH50", "ug_NO2"]
years = ["2030", "2050"]

OUTPUT_DIR = "data/2-output-data/maps_CHIMERE"
os.makedirs(OUTPUT_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(levelname)s - %(message)s")

# Global settings for map plots
plt.rcParams.update({
    "figure.dpi": 300,
    "font.size": 14,
    "axes.titlesize": 14,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12
})

# Load IRIS shapefile (polygons used for plotting)
shp_path = "data/1-processed-data/CONTOURS-IRIS/CONTOURS-IRIS.shp"
iris = gpd.read_file(shp_path)

# -------------------------
# Main plotting function (matrix: 2 rows scenarios x 3 cols [2019 conc, 2030 rel%, 2050 rel%])
# -------------------------
def plot_maps_for_pollutant(pol):
    pol_lbl = pretty_pol(pol)
    logging.info(f"Plotting matrix for {pol_lbl}: [2019 conc] [2030 conc + rel%] [2050 conc + rel%]")

    iris_code_col = _get_iris_code_col(iris)
    if iris_code_col is None:
        raise ValueError("IRIS shapefile does not contain a recognizable IRIS code column (e.g., CODE_IRIS).")

    # ---- Baseline points (2019) loaded once
    base_pts = coordo_ineris_chimere(pol, year="2019")

    data = {}
    all_conc_vals = []
    all_rel_vals = []
    extent = None

    # ---- Prepare population data aggregated at commune level (comcod) for each year
    pop_data = donnees_exportees_transformed.copy()
    pop_commune_by_year = {}
    for y in ["2019"] + years:
        pop_col = f"pop{y}"
        pop_data[pop_col] = pd.to_numeric(pop_data[pop_col], errors="coerce").fillna(0)
        pop_commune = pop_data.groupby("comcod")[pop_col].sum().reset_index()
        pop_commune.rename(columns={pop_col: "pop_val"}, inplace=True)
        pop_commune_by_year[y] = pop_commune

    def get_pw_mean_from_poly(target_poly_gdf, val_col, pop_commune):
        tmp = target_poly_gdf[[val_col, "comcod"]].copy()
        tmp[val_col] = pd.to_numeric(tmp[val_col], errors="coerce").astype(float)
        commune_vals = tmp.groupby("comcod")[val_col].mean().reset_index()
        merged = pd.merge(commune_vals, pop_commune, on="comcod", how="inner")
        total_pop = merged["pop_val"].sum()
        if total_pop == 0:
            return 0.0
        return float((merged[val_col] * merged["pop_val"]).sum() / total_pop)

    for sc in scenarios:
        logging.info(f"  Loading {sc.upper()}")
        data[sc] = {}

        # Use scenario CRS for polygons if available
        scen_pts_ref = coordo_chimere(pol, year=years[0], SC=sc.upper())
        iris_polys = iris.copy()
        if scen_pts_ref.crs is not None and iris_polys.crs != scen_pts_ref.crs:
            iris_polys = iris_polys.to_crs(scen_pts_ref.crs)

        # Attach comcod onto the IRIS polygons using donnees_exportees_transformed mapping
        iris_to_com = (
            donnees_exportees_transformed[["iriscod", "comcod"]]
            .drop_duplicates("iriscod")
            .rename(columns={"iriscod": iris_code_col})
        )
        iris_polys = iris_polys.merge(iris_to_com, on=iris_code_col, how="left")

        if extent is None:
            extent = _square_extent_from_bounds(iris_polys.total_bounds, pad_frac=0.02)

        # Baseline polygons (2019)
        base_poly = _points_to_iris_polygons(base_pts, "conc19", iris_polys, iris_code_col, fill_missing=True)
        base_poly = base_poly.rename(columns={"conc19": "conc_2019"})
        pw_mean_2019 = get_pw_mean_from_poly(base_poly, "conc_2019", pop_commune_by_year["2019"])

        data[sc]["2019"] = {
            "poly": base_poly,
            "pw_mean_conc": pw_mean_2019,
        }
        all_conc_vals.append(pd.to_numeric(base_poly["conc_2019"], errors="coerce").astype(float).values)

        for y in years:
            scen_pts = coordo_chimere(pol, year=y, SC=sc.upper())

            # Correction: returns delta_conc (scenario - baseline) by point
            corr_pts = correction_chimere(scen_pts, base_pts)

            scen_poly = _points_to_iris_polygons(scen_pts, "conc", iris_polys, iris_code_col, fill_missing=True)
            scen_poly = scen_poly.rename(columns={"conc": f"conc_{y}"})

            corr_poly = _points_to_iris_polygons(corr_pts, "delta_conc", iris_polys, iris_code_col, fill_missing=True)
            corr_poly = corr_poly.rename(columns={"delta_conc": f"delta_{y}"})

            merged = scen_poly[[iris_code_col, "comcod", f"conc_{y}", "geometry"]].merge(
                base_poly[[iris_code_col, "conc_2019"]],
                on=iris_code_col,
                how="left",
            ).merge(
                corr_poly[[iris_code_col, f"delta_{y}"]],
                on=iris_code_col,
                how="left",
            )

            denom = pd.to_numeric(merged["conc_2019"], errors="coerce").astype(float)
            delta = pd.to_numeric(merged[f"delta_{y}"], errors="coerce").astype(float)
            merged[f"rel_change_pct_{y}"] = np.where(denom != 0, (delta / denom) * 100.0, np.nan)

            pw_mean_conc = get_pw_mean_from_poly(merged, f"conc_{y}", pop_commune_by_year[y])
            pw_mean_rel = get_pw_mean_from_poly(merged, f"rel_change_pct_{y}", pop_commune_by_year[y])

            data[sc][y] = {
                "poly": merged,
                "pw_mean_conc": pw_mean_conc,
                "pw_mean_rel": pw_mean_rel,
            }

            all_conc_vals.append(pd.to_numeric(merged[f"conc_{y}"], errors="coerce").astype(float).values)
            all_rel_vals.append(pd.to_numeric(merged[f"rel_change_pct_{y}"], errors="coerce").astype(float).values)

    # ---- Shared color scales
    all_conc = np.concatenate(all_conc_vals) if len(all_conc_vals) else np.array([0.0, 1.0])
    conc_norm = Normalize(vmin=np.nanmin(all_conc), vmax=np.nanmax(all_conc))

    if len(all_rel_vals):
        all_rel = np.concatenate(all_rel_vals)
        rmax = np.nanmax(np.abs(all_rel)) if np.isfinite(all_rel).any() else 1.0
    else:
        rmax = 1.0
    rel_norm = TwoSlopeNorm(vcenter=0, vmin=-rmax, vmax=rmax)

    # ---- figure layout: 2 rows scenarios x 3 cols [2019 conc, 2030 rel%, 2050 rel%]
    fig, axes = plt.subplots(2, 3, figsize=(20, 11))
    fig.subplots_adjust(left=0.07, right=0.95, top=0.88, bottom=0.06, wspace=0.05, hspace=0.35)
    fig.suptitle(f"{pol_lbl} — 2019 / 2030 / 2050 (S2 and S3)", fontsize=18)

    for i, sc in enumerate(scenarios):
        # 2019 mean concentration (col 0)
        base_poly = data[sc]["2019"]["poly"]
        sm0 = fast_plot(
            axes[i, 0],
            base_poly,
            "conc_2019",
            f"{sc.upper()} — 2019 Mean Conc",
            "viridis",
            conc_norm,
            extent,
            pw_mean=data[sc]["2019"]["pw_mean_conc"],
            unit_label="µg/m³",
        )
        cbar_ax0 = axes[i, 0].inset_axes([1.02, 0.08, 0.035, 0.84])
        cb0 = plt.colorbar(sm0, cax=cbar_ax0)
        cb0.set_label("Mean Concentration (µg/m³)", fontsize=12)
        cb0.ax.tick_params(labelsize=12)

        # 2030: show relative change (%) in Blues; title includes mean conc + rel mean
        poly_2030 = data[sc]["2030"]["poly"]
        sm1 = fast_plot(
            axes[i, 1],
            poly_2030,
            "rel_change_pct_2030",
            f"{sc.upper()} — 2030 Mean Conc = {data[sc]['2030']['pw_mean_conc']:.2f} µg/m³\nRelative change = {data[sc]['2030']['pw_mean_rel']:.2f} %",
            "Blues",
            rel_norm,
            extent,
            pw_mean=None,
            unit_label="%",
        )
        cbar_ax1 = axes[i, 1].inset_axes([1.02, 0.08, 0.035, 0.84])
        cb1 = plt.colorbar(sm1, cax=cbar_ax1)
        cb1.set_label("Relative change (%)", fontsize=12)
        cb1.ax.tick_params(labelsize=12)

        # 2050: show relative change (%) in Blues; title includes mean conc + rel mean
        poly_2050 = data[sc]["2050"]["poly"]
        sm2 = fast_plot(
            axes[i, 2],
            poly_2050,
            "rel_change_pct_2050",
            f"{sc.upper()} — 2050 Mean Conc = {data[sc]['2050']['pw_mean_conc']:.2f} µg/m³\nRelative change = {data[sc]['2050']['pw_mean_rel']:.2f} %",
            "Blues",
            rel_norm,
            extent,
            pw_mean=None,
            unit_label="%",
        )
        cbar_ax2 = axes[i, 2].inset_axes([1.02, 0.08, 0.035, 0.84])
        cb2 = plt.colorbar(sm2, cax=cbar_ax2)
        cb2.set_label("Relative change (%)", fontsize=12)
        cb2.ax.tick_params(labelsize=12)

    # ---- Save
    out_path = os.path.join(OUTPUT_DIR, f"matrix_2019_2030_2050_{pol_lbl}_S2_S3.png")
    fig.savefig(out_path, dpi=300, bbox_inches="tight", pad_inches=0.05)
    plt.close(fig)
    logging.info(f"Saved: {out_path}")
# -------------------------
# Main driver
# -------------------------
if __name__ == "__main__":
    for pol in pollutants:
        plot_maps_for_pollutant(pol)

    logging.info("✅ All maps generated in 2x3 matrix format (2019 conc; 2030/2050 conc in title + rel% in Blues).")


2026-02-10 11:31:41,166 - INFO - INFO - Plotting matrix for PM2.5: [2019 conc] [2030 conc + rel%] [2050 conc + rel%]
2026-02-10 11:31:41,277 - INFO - INFO -   Loading S2


Starting coordo_ineris function
Loading data from data/1-processed-data/SHERPA/CHIMERE/outl.2019_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_ineris function
Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS2_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_chimere function


C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS2_FRA02_PM25_analysis_yravg.nc
Finished processing coordo_chimere function


C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


...

C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2030_scenS3_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function


C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Starting coordo_chimere function
Loading data from data\1-processed-data\SHERPA\CHIMERE\outl.2050_scenS3_FRA01_NO2_analysis_yravg.nc
Finished processing coordo_chimere function


C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
C:\Users\aysharma\anaconda3\Lib\site-packages\geopandas\array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
2026-02-10 11:33:46,388 - INFO - INFO - Saved: data/2-output-data/maps_CHIMERE\matrix_2019_2030_2050_NO2_S2_S3.png
2026-02-10 11:33:46,389 - INFO - INFO - ✅ All maps generated in 2x3 matrix format (2019 conc; 2030/2050 conc in title + rel% in Blues).
